In [1]:
import os
import pandas as pd
import numpy as np
import pickle
import torch
import json
from torch_geometric.data import HeteroData

In [25]:
DATABASE='drugbank'
DTI_PATH=f'../Data/DTI/{DATABASE}/dti_filter.pkl'
drug_features_dir=f"../Data/drug_features/{DATABASE}"
protein_features_dir=f"../Data/protein_features/{DATABASE}"
dti=pd.read_pickle(DTI_PATH)
drug_features = pd.read_pickle(f'{drug_features_dir}/drug_features_fused.pkl')
protein_features = pd.read_pickle(f'{protein_features_dir}/protein_features_fused.pkl')

In [26]:
dti.head(3)

,Drug,Protein
0,16129704,P00734
1,5311128,P22888
2,5311128,P30968


In [27]:
protein_features.head(3)

,Uniprot,Sequence,Concat_map,Embedding_256,Fusion_emb
0,P00734,MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRANT...,"[[-1.5557082, 4.3722734, 8.059093, -4.089855, ...","[7.3992224, -10.976476, -6.003735, -1.1471176,...","[7.3992224, -10.976476, -6.003735, -1.1471176,..."
1,P22888,MKQRFSALQLLKLLLLLQPPLPRALREALCPEPCNCVPDGALRCPG...,"[[0.85030776, 6.024752, -0.98376113, -0.754926...","[12.970119, -6.7147484, 2.3407416, 0.18203983,...","[12.970119, -6.7147484, 2.3407416, 0.18203983,..."
2,P30968,MANSASPEQNQNHCSAINNSIPLMQGNLPTLTLSGKIRVTVTFFLF...,"[[-0.5429804, 5.270295, 3.7761595, -2.558439, ...","[2.4458127, -10.027532, 15.077635, -0.53612053...","[2.4458127, -10.027532, 15.077635, -0.53612053..."


In [28]:
drug_features.head(3)

,PubChem,SMILES,SELFIES,Embedding,Embedding_256,ECFP4_128,Fusion_emb
0,16129704,CCC(C)C(C(=O)N1CCCC1C(=O)NC(CCC(=O)O)C(=O)NC(C...,[C][C][C][Branch1][C][C][C][Branch2][=Branch1]...,"[-0.077566646, -0.7603756, -2.0401356, -0.2728...","[6.332806, 7.4264226, 0.91277194, 2.9499185, 4...","[1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, ...","[6.332806, 7.4264226, 0.91277194, 2.9499185, 4..."
1,5311128,CC(C)CC(C(=O)NC(CCCN=C(N)N)C(=O)N1CCCC1C(=O)NN...,[C][C][Branch1][C][C][C][C][Branch2][Ring2][Br...,"[0.21207632, -0.82604843, -2.2601247, -0.30112...","[13.874359, 3.778419, 5.308585, 8.624356, 3.97...","[1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, ...","[13.874359, 3.778419, 5.308585, 8.624356, 3.97..."
2,45267103,CC(C)CC(C(=O)NC(C)C(=O)NC(C(C)C)C(=O)NC(C(C)C)...,[C][C][Branch1][C][C][C][C][Branch2][=N][O][C]...,"[-0.040166356, -0.8142738, -1.9942316, -0.6400...","[7.6426954, 7.4544473, 1.6735305, 5.621914, 5....","[1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, ...","[7.6426954, 7.4544473, 1.6735305, 5.621914, 5...."


In [29]:

DTI_PATH = f'../Data/DTI/{DATABASE}/dit_filter.pkl'
drug_features_dir = f"../Data/drug_features/{DATABASE}"
protein_features_dir = f"../Data/protein_features/{DATABASE}"
dti = pd.read_pickle(DTI_PATH)
drug_features = pd.read_pickle(f'{drug_features_dir}/drug_features_fused.pkl')
protein_features = pd.read_pickle(f'{protein_features_dir}/protein_features_fused.pkl')
dti.head(3)
protein_features.head(3)
drug_features.head(3)
# 创建映射字典
drug_mapping = {cid: i for i, cid in enumerate(drug_features['PubChem'])}
protein_mapping = {pid: i for i, pid in enumerate(protein_features['Uniprot'])}

# 构建边索引 - 只保留药物和蛋白质都在映射中的边
valid_edges = []
for drug, protein in zip(dti['Drug'], dti['Protein']):
    if drug in drug_mapping and protein in protein_mapping:
        valid_edges.append((drug_mapping[drug], protein_mapping[protein]))

src = [edge[0] for edge in valid_edges]
dst = [edge[1] for edge in valid_edges]

print(f"原始DTI记录数: {len(dti)}")
print(f"有效边数: {len(valid_edges)}")
print(f"过滤掉的记录数: {len(dti) - len(valid_edges)}")
print(f"src长度: {len(src)}, dst长度: {len(dst)}")

if len(valid_edges) == 0:
    raise ValueError("没有有效的边！请检查DTI中的Drug和Protein是否在features中存在")

edge_index = torch.tensor([src, dst])

# 创建异质图

output_dir = f'../Data/{DATABASE.upper()}'
# 提取 drug 和 protein 的嵌入
drug_x = torch.from_numpy(np.stack(drug_features['Fusion_emb'].values)).float()
protein_x = torch.from_numpy(np.stack(protein_features['Fusion_emb'].values)).float()

# 创建异质图
data = HeteroData()
data['drug'].x = drug_x
data['protein'].x = protein_x
data['drug', 'interaction', 'protein'].edge_index = edge_index
# 移动到 CUDA
device = 'cuda' if torch.cuda.is_available() else 'cpu'
data = data.to(device)
# 保存 mapping 和数据
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, 'drug_mapping.json'), 'w') as f:
    json.dump(drug_mapping, f)
with open(os.path.join(output_dir, 'protein_mapping.json'), 'w') as f:
    json.dump(protein_mapping, f)
torch.save(data, os.path.join(output_dir, f'hetero_data_{DATABASE.lower()}.pt'))
dti.to_pickle(os.path.join(output_dir, f'dti_{DATABASE.lower()}.pkl'))
with open(os.path.join(output_dir, f'info_dti_{DATABASE.lower()}.out'), 'w') as f:
    f.write(f'New shape after retrieved features: {dti.shape}\n')
    f.write(f'Unique drugs: {len(dti.Drug.unique())}\n')
    f.write(f'Unique proteins: {len(dti.Protein.unique())}\n')

原始DTI记录数: 23533
有效边数: 18784
过滤掉的记录数: 4749
src长度: 18784, dst长度: 18784


In [30]:
data

HeteroData(
  drug={ x=[5447, 384] },
  protein={ x=[4420, 384] },
  (drug, interaction, protein)={ edge_index=[2, 18784] }
)